# End-to-End Modeling — Home Credit

Full pipeline: **EDA → feature engineering → model tuning → save/load → predict** on `AMT_CREDIT` (loan amount).

- **Train:** `csv/application_train.csv`
- **Final observations:** `csv/application_test.csv`
- **Target:** `AMT_CREDIT` (regression)

## Part 1 — Extensive EDA

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

%matplotlib inline
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
CV_FOLDS = 3
PROJECT_ROOT = Path(r"C:\Users\User\Downloads\Machine Learning End to End Project\csv\application_train.csv")
TRAIN_PATH = PROJECT_ROOT / "csv" / "application_train.csv"
TEST_PATH = PROJECT_ROOT / "csv" / "application_test.csv"
MODELS_DIR = PROJECT_ROOT / "models"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
TARGET = "AMT_CREDIT"

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
print(f"Train shape: {df_train.shape}")
print(f"\nDtypes:\n{df_train.dtypes.value_counts()}")
df_train.head()

In [ ]:
missing_pct = (df_train.isnull().mean() * 100).sort_values(ascending=False)
missing_df = missing_pct.reset_index()
missing_df.columns = ["column", "pct_missing"]
print("Top 25 columns by % missing:")
display(missing_df.head(25))

top_missing = missing_pct.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_missing.values, y=top_missing.index, ax=ax, palette="viridis")
ax.set_xlabel("% Missing")
ax.set_title("Top 20 Columns by Missing %")
plt.tight_layout()
plt.show()

cols_over_50 = (missing_pct > 50).sum()
print(f"Columns with >50% missing: {cols_over_50}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_train[TARGET].dropna(), bins=50, kde=True, ax=axes[0])
axes[0].set_title(f"{TARGET} Distribution")
sns.boxplot(y=df_train[TARGET], ax=axes[1])
axes[1].set_title(f"{TARGET} Boxplot")
plt.tight_layout()
plt.show()

print(df_train[TARGET].describe())

In [ ]:
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in [TARGET, "SK_ID_CURR", "TARGET"]]

corr_with_target = df_train[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Top 15 numeric correlations with AMT_CREDIT:")
display(corr_with_target.head(15).to_frame("correlation"))

top_corr_cols = corr_with_target.head(10).index.tolist()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=corr_with_target.head(10).values, y=top_corr_cols, ax=ax, palette="coolwarm")
ax.set_xlabel("Correlation with AMT_CREDIT")
ax.set_title("Top 10 Numeric Correlations")
plt.tight_layout()
plt.show()

In [ ]:
cat_cols_eda = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE"]
for col in cat_cols_eda:
    print(f"\n=== {col} ===")
    print(df_train[col].value_counts(dropna=False).head(10))
    plt.figure(figsize=(8, 3))
    order = df_train[col].value_counts().head(8).index
    sns.boxplot(data=df_train, x=col, y=TARGET, order=order)
    plt.xticks(rotation=30, ha="right")
    plt.title(f"{TARGET} by {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
scatter_cols = ["AMT_INCOME_TOTAL", "AMT_GOODS_PRICE", "AMT_ANNUITY"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, scatter_cols):
    sample = df_train[[col, TARGET]].dropna().sample(min(3000, len(df_train)), random_state=RANDOM_STATE)
    sns.scatterplot(data=sample, x=col, y=TARGET, alpha=0.3, ax=ax)
    ax.set_title(f"{TARGET} vs {col}")
plt.tight_layout()
plt.show()

pair_cols = scatter_cols + [TARGET]
pair_sample = df_train[pair_cols].dropna().sample(min(1500, len(df_train)), random_state=RANDOM_STATE)
sns.pairplot(pair_sample, diag_kind="hist", corner=True, plot_kws={"alpha": 0.4, "s": 10})
plt.suptitle("Pairplot: Key Numerics vs AMT_CREDIT", y=1.02)
plt.show()

In [ ]:
def iqr_outlier_count(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lower) | (s > upper)).sum()), lower, upper

outlier_cols = [TARGET] + scatter_cols + ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
outlier_notes = []
for col in outlier_cols:
    if col in df_train.columns:
        n_out, lo, hi = iqr_outlier_count(df_train[col])
        outlier_notes.append({"column": col, "iqr_outliers": n_out, "lower_fence": lo, "upper_fence": hi})
outlier_df = pd.DataFrame(outlier_notes)
print("IQR-based outlier notes (1.5×IQR rule):")
display(outlier_df)
print("\nEDA insights:")
print("- AMT_CREDIT is right-skewed with high-value loans as upper outliers.")
print("- AMT_GOODS_PRICE and AMT_ANNUITY are strongly correlated with the target.")
print("- Many building/region columns have high missing rates; imputation is required.")
print("- DAYS_EMPLOYED uses sentinel 365243 for unemployed clients.")

## Part 2 — Feature Engineering & Preprocessing

In [ ]:
ID_COLS = ["SK_ID_CURR", "application_date"]
LEAKAGE_COLS = ["TARGET"]
UNEMPLOYED_SENTINEL = 365243
DAYS_PER_YEAR = 365.25

LOW_CARD_CAT = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "NAME_TYPE_SUITE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "WEEKDAY_APPR_PROCESS_START",
    "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "TOTALAREA_MODE",
    "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE",
]
HIGH_CARD_CAT = ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Add derived features without using AMT_CREDIT."""
    out = data.copy()
    out["AGE_YEARS"] = -out["DAYS_BIRTH"] / DAYS_PER_YEAR
    out["EMPLOYED_YEARS"] = np.where(
        out["DAYS_EMPLOYED"] == UNEMPLOYED_SENTINEL,
        0.0,
        -out["DAYS_EMPLOYED"] / DAYS_PER_YEAR,
    )
    out["GOODS_INCOME_RATIO"] = out["AMT_GOODS_PRICE"] / out["AMT_INCOME_TOTAL"].replace(0, np.nan)
    out["ANNUITY_INCOME_RATIO"] = out["AMT_ANNUITY"] / out["AMT_INCOME_TOTAL"].replace(0, np.nan)
    out["GOODS_TO_ANNUITY_RATIO"] = out["AMT_GOODS_PRICE"] / out["AMT_ANNUITY"].replace(0, np.nan)
    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    out["EXT_SOURCE_MEAN"] = out[ext_cols].mean(axis=1, skipna=True)
    out["EXT_SOURCE_MIN"] = out[ext_cols].min(axis=1, skipna=True)
    out["EXT_SOURCE_MAX"] = out[ext_cols].max(axis=1, skipna=True)
    out["INCOME_PER_CHILD"] = out["AMT_INCOME_TOTAL"] / (out["CNT_CHILDREN"] + 1)
    out["REGISTRATION_YEARS"] = -out["DAYS_REGISTRATION"] / DAYS_PER_YEAR
    out["ID_PUBLISH_YEARS"] = -out["DAYS_ID_PUBLISH"] / DAYS_PER_YEAR
    return out


class E2EPreprocessor:
    """Fit/transform pipeline for train and inference."""

    def __init__(self):
        self.freq_maps_ = {}
        self.numeric_cols_ = None
        self.low_card_cols_ = None
        self.high_card_cols_ = None
        self.numeric_medians_ = {}
        self.low_card_modes_ = {}
        self.feature_names_ = None
        self.dropped_columns_ = []
        self.used_raw_columns_ = []
        self.engineered_columns_ = [
            "AGE_YEARS", "EMPLOYED_YEARS", "GOODS_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
            "GOODS_TO_ANNUITY_RATIO", "EXT_SOURCE_MEAN", "EXT_SOURCE_MIN", "EXT_SOURCE_MAX",
            "INCOME_PER_CHILD", "REGISTRATION_YEARS", "ID_PUBLISH_YEARS",
        ]

    def _select_raw_columns(self, df):
        drop = set(ID_COLS + LEAKAGE_COLS + [TARGET])
        self.dropped_columns_ = sorted([c for c in drop if c in df.columns])
        self.used_raw_columns_ = [c for c in df.columns if c not in drop]
        return df[self.used_raw_columns_].copy()

    def fit(self, df):
        work = engineer_features(self._select_raw_columns(df))
        self.numeric_cols_ = [
            c for c in work.columns
            if c not in LOW_CARD_CAT + HIGH_CARD_CAT
            and work[c].dtype in [np.float64, np.float32, np.int64, np.int32, np.int16, np.uint8, bool]
        ]
        self.low_card_cols_ = [c for c in LOW_CARD_CAT if c in work.columns]
        self.high_card_cols_ = [c for c in HIGH_CARD_CAT if c in work.columns]

        for col in self.numeric_cols_:
            self.numeric_medians_[col] = work[col].median()
        for col in self.low_card_cols_:
            mode_val = work[col].mode(dropna=True)
            self.low_card_modes_[col] = mode_val.iloc[0] if len(mode_val) else "Unknown"
        for col in self.high_card_cols_:
            freq = work[col].value_counts(normalize=True, dropna=False)
            self.freq_maps_[col] = freq.to_dict()

        _ = self.transform(df)
        return self

    def transform(self, df):
        work = engineer_features(self._select_raw_columns(df))
        for col, val in self.numeric_medians_.items():
            work[col] = work[col].fillna(val)
        for col, val in self.low_card_modes_.items():
            work[col] = work[col].fillna(val)
        for col in self.high_card_cols_:
            work[col] = work[col].map(self.freq_maps_[col]).fillna(0.0)

        low_dummies = pd.get_dummies(work[self.low_card_cols_], prefix=self.low_card_cols_, drop_first=True)
        high_df = work[self.high_card_cols_]
        X = pd.concat([work[self.numeric_cols_], high_df, low_dummies], axis=1)

        if self.feature_names_ is None:
            self.feature_names_ = X.columns.tolist()
        else:
            for col in self.feature_names_:
                if col not in X.columns:
                    X[col] = 0.0
            X = X[self.feature_names_]
        return X.astype(float)

    def fit_transform(self, df):
        return self.fit(df).transform(df)


preprocessor = E2EPreprocessor()
X_full = preprocessor.fit_transform(df_train)
y_full = df_train[TARGET].copy()

print(f"Raw columns used: {len(preprocessor.used_raw_columns_)}")
print(f"Dropped columns: {preprocessor.dropped_columns_}")
print(f"Engineered columns added: {len(preprocessor.engineered_columns_)}")
print(f"Final feature count: {len(preprocessor.feature_names_)}")
print(f"\nSample feature names (first 15): {preprocessor.feature_names_[:15]}")
X_full.head()

## Part 3 — Train/Test Split

In [ ]:
X_model = X_full.copy()
y_model = y_full.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Hold-out test: {X_test.shape[0]:,} rows")
print(f"Final observations (application_test.csv) kept separate for Part 7.")

## Part 4 — All Models with GridSearchCV

In [ ]:
def evaluate_model(model, X_te, y_te):
    pred = model.predict(X_te)
    return {
        "R2": r2_score(y_te, pred),
        "MAE": mean_absolute_error(y_te, pred),
        "RMSE": np.sqrt(mean_squared_error(y_te, pred)),
    }


def run_search(name, search, X_tr, y_tr):
    print(f"Tuning {name}...")
    search.fit(X_tr, y_tr)
    print(f"  Best params: {search.best_params_}")
    print(f"  Best CV neg_MSE: {search.best_score_:,.0f}\n")
    return search.best_estimator_, search.best_params_


tuned_models = {}
best_params = {}

# 1. Linear Regression (baseline)
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
lr_pipe.fit(X_train, y_train)
tuned_models["Linear Regression"] = lr_pipe
best_params["Linear Regression"] = "default (no tuning)"
print("Linear Regression: fitted baseline\n")

# 2. Ridge
ridge_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    param_grid={"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=CV_FOLDS, scoring="neg_mean_squared_error", n_jobs=-1,
)
tuned_models["Ridge"], best_params["Ridge"] = run_search("Ridge", ridge_search, X_train, y_train)

# 3. Lasso
lasso_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10000))]),
    param_grid={"model__alpha": [0.001, 0.01, 0.1, 1, 10]},
    cv=CV_FOLDS, scoring="neg_mean_squared_error", n_jobs=-1,
)
tuned_models["Lasso"], best_params["Lasso"] = run_search("Lasso", lasso_search, X_train, y_train)

# 4. ElasticNet
en_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10000))]),
    param_grid={
        "model__alpha": [0.001, 0.01, 0.1, 1, 10],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
    },
    cv=CV_FOLDS, scoring="neg_mean_squared_error", n_jobs=-1,
)
tuned_models["ElasticNet"], best_params["ElasticNet"] = run_search("ElasticNet", en_search, X_train, y_train)

# 5. Decision Tree
dt_search = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [5, 10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 5, 10],
    },
    cv=CV_FOLDS, scoring="neg_mean_squared_error", n_jobs=-1,
)
tuned_models["Decision Tree"], best_params["Decision Tree"] = run_search("Decision Tree", dt_search, X_train, y_train)

# 6. Random Forest
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [50, 100, 150, 200],
        "max_depth": [8, 12, 16, 20, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    n_iter=30, cv=CV_FOLDS, scoring="neg_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=-1,
)
tuned_models["Random Forest"], best_params["Random Forest"] = run_search("Random Forest", rf_search, X_train, y_train)

# 7. Gradient Boosting
gbr_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_distributions={
        "n_estimators": [50, 100, 150],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "min_samples_leaf": [1, 2, 5],
    },
    n_iter=30, cv=CV_FOLDS, scoring="neg_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=-1,
)
tuned_models["Gradient Boosting"], best_params["Gradient Boosting"] = run_search(
    "Gradient Boosting", gbr_search, X_train, y_train
)

## Part 5 — Model Comparison

In [ ]:
results = []
for name, model in tuned_models.items():
    metrics = evaluate_model(model, X_test, y_test)
    results.append({
        "Model": name,
        "Best Params": best_params[name],
        "R2": metrics["R2"],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
    })

results_df = pd.DataFrame(results).sort_values(["R2", "RMSE"], ascending=[False, True]).reset_index(drop=True)
results_df_display = results_df.rename(columns={"R2": "R²"})
display(results_df_display)

best_row = results_df.iloc[0]
best_name = best_row["Model"]
best_model = tuned_models[best_name]

print("=" * 60)
print(f"BEST MODEL: {best_name}")
print(f"  Params: {best_row['Best Params']}")
print(f"  R²:   {best_row['R2']:.4f}")
print(f"  MAE:  {best_row['MAE']:,.2f}")
print(f"  RMSE: {best_row['RMSE']:,.2f}")
print("=" * 60)

## Part 6 — Save & Load Models

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_filenames = {
    "Linear Regression": "e2e_linear_regression.joblib",
    "Ridge": "e2e_ridge.joblib",
    "Lasso": "e2e_lasso.joblib",
    "ElasticNet": "e2e_elasticnet.joblib",
    "Decision Tree": "e2e_decision_tree.joblib",
    "Random Forest": "e2e_random_forest.joblib",
    "Gradient Boosting": "e2e_gradient_boosting.joblib",
}

saved_paths = []
for name, fname in model_filenames.items():
    path = MODELS_DIR / fname
    joblib.dump(tuned_models[name], path)
    saved_paths.append(str(path))
    print(f"Saved {name} -> {path}")

best_model_path = MODELS_DIR / "e2e_best_model.joblib"
joblib.dump(best_model, best_model_path)
saved_paths.append(str(best_model_path))
print(f"\nSaved best model -> {best_model_path}")

preprocessor_path = MODELS_DIR / "e2e_preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)
saved_paths.append(str(preprocessor_path))
print(f"Saved preprocessor -> {preprocessor_path}")

feature_names_path = MODELS_DIR / "e2e_feature_names.json"
with open(feature_names_path, "w", encoding="utf-8") as f:
    json.dump(preprocessor.feature_names_, f, indent=2)
saved_paths.append(str(feature_names_path))

metadata = {
    "target": TARGET,
    "n_features": len(preprocessor.feature_names_),
    "dropped_columns": preprocessor.dropped_columns_,
    "engineered_columns": preprocessor.engineered_columns_,
    "best_model_name": best_name,
    "models": [],
}
for _, row in results_df.iterrows():
    params = row["Best Params"]
    if not isinstance(params, dict):
        params = str(params)
    metadata["models"].append({
        "model_name": row["Model"],
        "filename": model_filenames[row["Model"]],
        "best_hyperparameters": params,
        "r2": float(row["R2"]),
        "mae": float(row["MAE"]),
        "rmse": float(row["RMSE"]),
    })

metadata_path = MODELS_DIR / "e2e_model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
saved_paths.append(str(metadata_path))
print(f"Saved metadata -> {metadata_path}")

In [ ]:
loaded_best = joblib.load(best_model_path)
loaded_preprocessor = joblib.load(preprocessor_path)
sample_pred = loaded_best.predict(X_test.head(5))
print(f"Loaded best model type: {type(loaded_best).__name__}")
print(f"Loaded preprocessor feature count: {len(loaded_preprocessor.feature_names_)}")
print(f"Sample predictions: {sample_pred}")

## Part 7 — Predict Final Observations

In [ ]:
df_final = pd.read_csv(TEST_PATH)
print(f"Final observations shape: {df_final.shape}")

X_final = loaded_preprocessor.transform(df_final)
final_predictions = loaded_best.predict(X_final)

pred_df = pd.DataFrame({
    "SK_ID_CURR": df_final["SK_ID_CURR"],
    "AMT_CREDIT": final_predictions,
})

print("Sample predictions (head 10):")
display(pred_df.head(10))

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
predictions_path = PREDICTIONS_DIR / "amtcredit_predictions.csv"
pred_df.to_csv(predictions_path, index=False)
print(f"\nSaved predictions -> {predictions_path}")
print(f"Prediction stats: mean={pred_df['AMT_CREDIT'].mean():,.2f}, "
      f"min={pred_df['AMT_CREDIT'].min():,.2f}, max={pred_df['AMT_CREDIT'].max():,.2f}")

In [ ]:
print("File existence check:")
check_files = saved_paths + [str(predictions_path)]
for p in check_files:
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p) / 1024 if exists else 0
    print(f"  {'OK' if exists else 'MISSING'}  {p}  ({size_kb:.1f} KB)")

## Part 8 — Conclusion

### Best model
The best tuned model (by hold-out **R²**) is reported in Part 5. Tree ensembles typically outperform linear baselines when many numeric and encoded categorical features are used.

### EDA insights
- **AMT_CREDIT** is right-skewed; **AMT_GOODS_PRICE** and **AMT_ANNUITY** are the strongest numeric predictors.
- Many property/region columns have high missingness; median imputation was applied.
- High-cardinality fields (**ORGANIZATION_TYPE**, **OCCUPATION_TYPE**) were frequency-encoded to avoid explosion of dummy variables.

### Next steps
1. Try target log-transform for skew reduction.
2. Add bureau / previous-application aggregates for richer features.
3. Use cross-validated stacking or LightGBM/XGBoost for further gains.
4. Calibrate prediction intervals for risk-aware lending scenarios.